## Querying Snowflake Metadata Commands (SHOW, DESCRIBE, LIST)

---

### The Problem

Snowflake metadata commands (`SHOW`, `DESCRIBE`, `LIST`) are NOT regular SQL queries. You **cannot** use them as subqueries:

```sql
-- THIS DOES NOT WORK
SELECT * FROM (SHOW TABLES IN SCHEMA my_schema);  -- ERROR
```

These commands produce output, but the output is not a table you can query directly.

---

### Two Solutions

**1. RESULT_SCAN() — Query the output of the last executed command**

```sql
-- Step 1: Run the metadata command
SHOW TABLES IN SCHEMA my_db.my_schema;

-- Step 2: Query its output using RESULT_SCAN
SELECT "name", "rows", "bytes"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
```

`RESULT_SCAN()` converts the output of any previously executed statement into a queryable result set.

**2. Pipe Operator (->>) — Chain the command inline**

```sql
-- Single statement: run SHOW and query its output
SHOW TABLES IN SCHEMA my_db.my_schema
  ->> SELECT "name", "rows", "bytes" FROM $1 WHERE "rows" > 1000;
```

The pipe operator (`->>`) passes the output of the left side as input to the right side. You reference the previous result using `$1` in the `FROM` clause.

**How $1 works in pipe operator:**
- `$1` = result of the immediately preceding statement
- `$2` = result of 2 statements back
- `$n` = result of n statements back
- `$n` is **only valid in the FROM clause**

```sql
-- Chaining multiple statements
SELECT * FROM dept WHERE dname = 'SALES'
  ->> SELECT * FROM emp WHERE deptno IN (SELECT deptno FROM $1)
  ->> SELECT ename, sal FROM $1 ORDER BY sal DESC;
```

---

### Why Double Quotes?

SHOW command output columns are **lowercase** (e.g., `name`, `rows`, `bytes`, `created_on`). Snowflake identifiers are uppercase by default. Without double quotes, Snowflake looks for `NAME` but the column is `name` — mismatch.

```sql
-- WRONG: Snowflake looks for column NAME (uppercase) — not found
SELECT name FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

-- CORRECT: Double quotes preserve lowercase
SELECT "name" FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
```

---

### What is $1, $2, $3 in Different Contexts?

`$n` means different things depending on where it's used:

**1. In Pipe Operator (FROM clause) — refers to a previous statement's result:**
```sql
SHOW WAREHOUSES
  ->> SELECT "name", "state" FROM $1;  -- $1 = output of SHOW WAREHOUSES
```

**2. In SELECT list — positional column reference (1-based):**
```sql
SHOW WAREHOUSES;

-- $1 = first column, $2 = second column, etc.
SELECT $1 AS warehouse_name, $2 AS state, $4 AS size
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
```

**3. In COPY INTO — refers to fields in the source file:**
```sql
-- $1 = first field in CSV, $2 = second field, etc.
COPY INTO my_table(name, age)
FROM (SELECT $1, $2 FROM @my_stage)
FILE_FORMAT = (TYPE = CSV);
```

---

### Complete Examples

**Example 1: Find all tables with more than 1000 rows (RESULT_SCAN)**
```sql
SHOW TABLES IN SCHEMA my_db.public;

SELECT "name", "database_name", "schema_name", "rows"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "rows" > 1000
ORDER BY "rows" DESC;
```

**Example 2: List running warehouses (Pipe Operator)**
```sql
SHOW WAREHOUSES
  ->> SELECT "name", "state", "size" FROM $1 WHERE "state" = 'STARTED';
```

**Example 3: Tables created after a date (Pipe Operator)**
```sql
SHOW TABLES
  ->> SELECT "created_on" AS creation_date,
             "name" AS table_name,
             "owner" AS table_owner
       FROM $1
       WHERE creation_date > '2025-04-15'::DATE;
```

**Example 4: Filter SHOW GRANTS using positional column references**
```sql
SHOW GRANTS TO ROLE analyst;

SELECT $2 AS privilege, $4 AS granted_on, $5 AS object_name
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
```

---

### Summary

| Method | Syntax | Statements | FROM needed? |
|--------|--------|------------|-------------|
| RESULT_SCAN | `SELECT ... FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))` | Two (SHOW + SELECT) | YES — `FROM TABLE(RESULT_SCAN(...))` |
| Pipe Operator | `SHOW ... ->> SELECT ... FROM $1` | One (combined) | YES — `FROM $1` |

**Key rules:**
- Always use `"double_quotes"` for column names from SHOW/DESCRIBE output
- Pipe operator `$n` in FROM = reference to previous statement's result
- `$n` in SELECT list = positional column reference (1-based)
- RESULT_SCAN works with ANY previous statement (not just SHOW)
- Pipe operator is `->>` (not `=>>`) and the chain ends with a single semicolon
- No semicolons between chained statements — only at the very end

### Multi-Statement Pipe Chains

`$n` counts **backward** from the current statement:
- `$1` = result of the immediately preceding statement
- `$2` = result of 2 statements back
- `$3` = result of 3 statements back

```
Statement A              ← referenced as $3 in D, $2 in C, $1 in B
  ->> Statement B        ← referenced as $2 in D, $1 in C
  ->> Statement C        ← referenced as $1 in D
  ->> Statement D;       ← final output returned to client
```

**Example: Join results from two different queries**
```sql
SELECT * FROM dept WHERE dname = 'SALES'                              -- Statement 1
  ->> SELECT * FROM emp WHERE deptno IN (SELECT deptno FROM $1)        -- Statement 2 ($1 = dept result)
  ->> SELECT ename, sal FROM $1 ORDER BY sal DESC;                     -- Statement 3 ($1 = emp result)
```

**Example: Count rows inserted across multiple INSERTs**
```sql
CREATE OR REPLACE TABLE t (a INT, b INT)
  ->> INSERT INTO t VALUES (1, 2)       -- $4 from last statement
  ->> INSERT INTO t VALUES (3, 4)       -- $3 from last statement
  ->> INSERT INTO t VALUES (5, 6)       -- $2 from last statement
  ->> INSERT INTO t VALUES (7, 8)       -- $1 from last statement
  ->> SELECT (SELECT $1 FROM $4) +
             (SELECT $1 FROM $3) +
             (SELECT $1 FROM $2) +
             (SELECT $1 FROM $1) AS "Total rows inserted";
```

> Note: `$1` in `SELECT $1 FROM $4` — the `$1` in SELECT is a positional column reference (first column of the result), while `$4` in FROM refers to the 4th statement back.

### GET_DDL() vs DESCRIBE

---

**GET_DDL()** returns the complete SQL statement to recreate an object. **DESCRIBE** shows column metadata only.

| | GET_DDL() | DESCRIBE (DESC) |
|--|-----------|----------------|
| Returns | Full CREATE statement | Column names, types, nullable, defaults |
| Includes constraints, clustering keys, comments | YES | NO |
| Use case | Recreate/migrate objects | Quick column inspection |

---

### GET_DDL() Syntax

```sql
SELECT GET_DDL('<object_type>', '<fully_qualified_name>');
```

**Examples:**
```sql
-- Table
SELECT GET_DDL('TABLE', 'SALES_DB.PUBLIC.ORDERS');

-- View
SELECT GET_DDL('VIEW', 'MY_DB.PUBLIC.MY_VIEW');

-- Schema (includes all objects in it)
SELECT GET_DDL('SCHEMA', 'MY_DB.MY_SCHEMA');

-- Database (includes all schemas and objects)
SELECT GET_DDL('DATABASE', 'MY_DB');

-- Stage
SELECT GET_DDL('STAGE', 'MY_DB.PUBLIC.MY_STAGE');

-- File Format
SELECT GET_DDL('FILE FORMAT', 'MY_DB.PUBLIC.MY_FF');
```

**Sample output for a table:**
```sql
CREATE OR REPLACE TABLE ORDERS (
    ORDER_ID NUMBER,
    CUSTOMER_ID NUMBER,
    ORDER_DATE DATE
);
```

---

### DESCRIBE Syntax

```sql
DESCRIBE TABLE MY_DB.MY_SCHEMA.MY_TABLE;
-- or
DESC TABLE MY_DB.MY_SCHEMA.MY_TABLE;
```

Shows metadata (column name, type, nullable, default, etc.) but NOT the full CREATE statement.

---

### Generate DDL for All Tables in a Schema

```sql
SELECT table_name,
       GET_DDL('TABLE', table_catalog || '.' || table_schema || '.' || table_name) AS ddl
FROM INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'PUBLIC';
```

---

**Exam Tip:** GET_DDL() is the preferred approach when you need the actual creation script. DESC TABLE only gives you column metadata.